### Processing POEM results
This notebook takes the output from POEM, specificaly the output named "input.fsa.operon" that has the information about the operon genes in each BGC.

In [1]:
import pandas as pd
import re

def process_poem_table(table_path, sep="\t"):

    df = pd.read_csv(table_path, sep=sep)
    df.columns = df.columns.str.strip().str.lower()

    if "gene_id" not in df.columns:
        raise ValueError(f"Coluna 'gene_id' não encontrada. Colunas disponíveis: {list(df.columns)}")

    def parse_operon_string(s):
        genes = re.split(r'-->|<--', str(s))

        metagenome = None
        strand = None
        coords = []

        for g in genes:
            m = re.search(r'\|([+-])\|(\d+)\|(\d+)\$\$(MGYG\d+_\d+)', g)
            if m:
                strand = m.group(1)
                start = m.group(2)
                end = m.group(3)
                metagenome = m.group(4)
                coords.append(f"{start}/{end}")

        if strand == "+":
            coord_string = " --> ".join(coords)
        else:
            coord_string = " <-- ".join(coords)

        return pd.Series([metagenome, strand, coord_string])

    df[["metagenome_id", "strand", "coordinates"]] = df["gene_id"].apply(parse_operon_string)

    return df

In [2]:
df = process_poem_table("/home/pedro/resultados_bgc/fna_dos_bgcs/BGCS.fasta_output/input.fsa.operon")
df

,gene_id,cog_id,annotation,metagenome_id,strand,coordinates
0,PROKKA_00927|Prodigal:2.6|2342_aa|+|2227|3084$...,unknown-->unknown-->unknown-->unknown-->unknow...,unknown::unknown-->unknown::unknown-->unknown:...,MGYG000296006_71,+,2227/3084 --> 3081/3860 --> 3873/4478 --> 4475...
1,PROKKA_01473|Prodigal:2.6|3269_aa|+|3329|4378$...,unknown-->unknown-->unknown-->unknown-->unknow...,unknown::unknown-->unknown::unknown-->unknown:...,MGYG000296008_2,+,3329/4378 --> 4378/5151 --> 5102/5692 --> 5734...
2,PROKKA_01469|Prodigal:2.6|920_aa|-|494|1084$$M...,unknown<--unknown<--unknown<--unknown,unknown::unknown<--unknown::unknown<--unknown:...,MGYG000296008_2,-,494/1084 <-- 1065/1799 <-- 1803/2372 <-- 2373/...
3,PROKKA_01481|Prodigal:2.6|9060_aa|-|12714|1329...,unknown<--unknown<--unknown<--unknown<--unknow...,unknown::unknown<--unknown::unknown<--unknown:...,MGYG000296008_2,-,12714/13298 <-- 13295/13717 <-- 13756/14715 <-...
4,PROKKA_01452|Prodigal:2.6|1655_aa|-|206|1723$$...,unknown<--unknown<--unknown<--unknown<--unknow...,unknown::unknown<--unknown::unknown<--unknown:...,MGYG000296008_22,-,206/1723 <-- 1710/2171 <-- 2164/2628 <-- 2638/...
...,...,...,...,...,...,...
366,PROKKA_01377|Prodigal:2.6|8178_aa|-|10481|1167...,unknown<--unknown,unknown::unknown<--unknown::unknown,MGYG000296076_4,-,10481/11671 <-- 11671/12852
367,PROKKA_01383|Prodigal:2.6|12951_aa|-|18379|190...,unknown<--unknown,unknown::unknown<--unknown::unknown,MGYG000296076_4,-,18379/19077 <-- 19088/19942
368,PROKKA_01388|Prodigal:2.6|15803_aa|-|22959|234...,unknown<--unknown<--unknown,unknown::unknown<--unknown::unknown<--unknown:...,MGYG000296076_4,-,22959/23456 <-- 23503/24327 <-- 24362/24787
369,PROKKA_01395|Prodigal:2.6|18981_aa|-|27735|282...,unknown<--unknown,unknown::unknown<--unknown::unknown,MGYG000296076_4,-,27735/28226 <-- 28236/28712


In [5]:
df = df[["metagenome_id", "strand", "coordinates"]]
df

,metagenome_id,strand,coordinates
0,MGYG000296006_71,+,2227/3084 --> 3081/3860 --> 3873/4478 --> 4475...
1,MGYG000296008_2,+,3329/4378 --> 4378/5151 --> 5102/5692 --> 5734...
2,MGYG000296008_2,-,494/1084 <-- 1065/1799 <-- 1803/2372 <-- 2373/...
3,MGYG000296008_2,-,12714/13298 <-- 13295/13717 <-- 13756/14715 <-...
4,MGYG000296008_22,-,206/1723 <-- 1710/2171 <-- 2164/2628 <-- 2638/...
...,...,...,...
366,MGYG000296076_4,-,10481/11671 <-- 11671/12852
367,MGYG000296076_4,-,18379/19077 <-- 19088/19942
368,MGYG000296076_4,-,22959/23456 <-- 23503/24327 <-- 24362/24787
369,MGYG000296076_4,-,27735/28226 <-- 28236/28712


In [6]:
df.to_csv("POEM_operon_information.csv")